# 08 — 시점 정합성 점검 + nowcast + 채널 스냅샷 + 라벨 시뮬레이션

07(리더보드 **0.5917**, 사내 검증 0.559)을 기반으로 `TASK_08_nowcast.md` 지침의 단계 A~E를 구현하고 비교한다.

> ⚠ **운영진 확인 필요** — 07과 마찬가지로 공식 `train_labels.csv`의 **08-25 라벨**을 피처로 쓴다.
> 이 라벨은 09-01 이후 영상 성과로 계산된 값이라 "기준일 이후 정보"에 해당한다. 사용 전 운영진 확인을 권장한다.

**지침 전제에 대한 사전 점검 결과 (중요)**
지침은 "테스트(09-01) 시점에는 08-26 12시 이후 업로드 영상의 view_5d가 없다"고 가정한다.
실제 배포 `video_5d_views`에는 기준일 직전 132시간 안에 올라온 롱폼의 **약 96%가 view_5d를 갖고 있고**,
이 비율은 학습 기준일(08-01, 08-05)과 테스트(09-01)가 같다(§0 표).
따라서 이 노트북은 지침의 단계 A(132h 마스킹)·B(nowcast)를 **구현하되, 마스킹하지 않은 07 방식(`full`)과 나란히 검증**해
더 나은 쪽을 채택한다.

| 모드 | view_5d 사용 범위 |
|---|---|
| `full` (07 방식) | `published_at < Dref`의 실제 view_5d 전부 |
| `trunc` (단계 A) | `published_at < Dref − 132h`의 실제 view_5d만 |
| `nowcast` (단계 B) | `trunc` + `[Dref − 132h, Dref)` 영상은 초기 스냅샷으로 추정한 view_5d |

In [1]:
# ===== 라이브러리 =====
import os, glob, warnings, time
import duckdb
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from sklearn.inspection import permutation_importance
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

# ===== 경로 · 상수 =====
RAW  = os.environ.get('EPOCH_RAW', '../data/raw/epoch_data')
PROC = '../data/processed'                 # 캐시 저장 위치
SUB  = os.environ.get('EPOCH_SUB', '../submissions')
D, PREV = '2026-09-01', '2026-08-25'       # 본선 기준일 / 전주 공식 라벨 기준일
FUT, FUT_PREV = 26, 18                     # 본선 / 공식 라벨 정답 구간 길이 (일)
LAG_H = 132                                # view_5d 확정까지 걸리는 시간 (업로드 후 132h)
SEED = 42
EVAL_DATES = ['2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04', '2026-08-05']   # 07과 같은 검증 기준일
TRAIN_DATES = pd.date_range('2026-07-26', '2026-08-05', freq='D').strftime('%Y-%m-%d').tolist()

# video_snapshots 원본: 레포의 parquet은 Git LFS 포인터(134B)라 실제 파일이 없으면 다운로드한 CSV를 쓴다
SNAP_SRC = os.environ.get('EPOCH_SNAP', f'{PROC}/video_snapshots.parquet')
if not os.path.exists(SNAP_SRC) or os.path.getsize(SNAP_SRC) < 10_000:
    SNAP_SRC = os.path.expanduser('~/Downloads/video_snapshots.csv')
EARLY = f'{PROC}/cache_08_snap_early.parquet'   # 롱폼 · 업로드 후 132h 이내 스냅샷만 추린 캐시
RATIO = f'{PROC}/cache_08_ratio.parquet'        # nowcast 비율 테이블 캐시

# ===== DuckDB: CSV를 파일 이름 그대로 뷰로 등록 =====
con = duckdb.connect()
con.sql("SET memory_limit = '4GB'")
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    con.sql(f"CREATE OR REPLACE VIEW {os.path.splitext(os.path.basename(p))[0]} AS "
            f"SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")

# 1,800만 행 video_snapshots는 pandas로 읽지 않고, DuckDB에서 필요한 부분(롱폼 · 132h 이내)만 캐시로 저장
if not os.path.exists(EARLY):
    src = f"'{SNAP_SRC}'" if SNAP_SRC.endswith('.parquet') else f"read_csv('{SNAP_SRC}', header=true, auto_detect=true)"
    con.sql(f"""COPY (
        SELECT s.video_id, s.channel_id, v.published_at, s.collected_at, s.video_age_hours, s.view_count
        FROM {src} s
        JOIN (SELECT video_id, published_at FROM videos WHERE is_shorts = 0) v USING (video_id)
        WHERE s.video_age_hours <= {LAG_H}
    ) TO '{EARLY}' (FORMAT parquet)""")
con.sql(f"CREATE OR REPLACE VIEW snap_early AS SELECT * FROM '{EARLY}'")
print('초기 스냅샷 캐시:', con.sql("SELECT COUNT(*), COUNT(DISTINCT video_id) FROM snap_early").fetchone())

초기 스냅샷 캐시: (9257337, 22217)


## 0. 사전 점검 — 기준일 직전 132h 영상의 view_5d가 정말 없는가

지침 전제를 그대로 따르기 전에 데이터로 확인한다.
마지막 열 `h_after`는 view_5d를 만든 스냅샷이 업로드 후 몇 시간 시점인지다(≈120h면 정상 측정).

In [2]:
pre = []
for d in ['2026-08-01', '2026-08-05', D]:
    n, k, h = con.sql(f"""
        SELECT COUNT(*), COUNT(f.view_5d), ROUND(AVG(f.h_after), 1)
        FROM videos v LEFT JOIN video_5d_views f USING (video_id)
        WHERE v.is_shorts = 0
          AND v.published_at >= TIMESTAMP '{d}' - INTERVAL {LAG_H} HOUR AND v.published_at < TIMESTAMP '{d}'
    """).fetchone()
    pre.append({'기준일': d, '직전 132h 롱폼': n, 'view_5d 있음': k, '비율': round(k / n, 3), 'h_after 평균': h})
display(pd.DataFrame(pre))
print('video_snapshots 최대 수집 시각:', con.sql("SELECT MAX(collected_at) FROM snap_early").fetchone()[0])

,기준일,직전 132h 롱폼,view_5d 있음,비율,h_after 평균
0,2026-08-01,2598,2511,0.967,120.3
1,2026-08-05,2568,2473,0.963,120.1
2,2026-09-01,2945,2825,0.959,120.1


video_snapshots 최대 수집 시각: 2026-08-31 23:45:54


→ 학습 기준일과 09-01 모두 약 96%가 view_5d를 갖고 있다. 08-26 이후 영상의 view_5d는 09-01 이후 스냅샷(배포 안 됨)으로 계산되어 함께 배포되었다.
즉 07의 `full` 방식은 이미 학습·테스트가 같은 조건이다. 아래 단계 A·B는 이 사실을 **검증으로 확인**하는 비교 실험이다.

## 1. 공통 함수

- `nowcast(Dref)`: `[Dref−132h, Dref)` 롱폼의 view_5d를 `collected_at < Dref` 마지막 스냅샷 × 나이별 보정비율로 추정
- `v5_view(Dref, mode)`: 모드별 view_5d 테이블 (실제 + 추정, `is_hat` 표시). **학습·테스트 모두 이 함수 하나로**
- `make_features`, `add_prev_features`: 07과 같은 피처를 `v5_view` 위에서 계산

In [3]:
# ===== nowcast 비율 테이블 (한 번만 계산해 캐시) =====
# 영상마다 6시간 나이 구간(6~108h)의 마지막 스냅샷 조회수와 실제 view_5d의 log 비율
if not os.path.exists(RATIO):
    con.sql(f"""COPY (
        WITH b AS (
            SELECT video_id, channel_id, published_at,
                   CAST(FLOOR(video_age_hours / 6) * 6 AS INTEGER) AS age_b,
                   ARG_MAX(view_count, collected_at) AS v_b              -- 구간 내 마지막 스냅샷
            FROM snap_early
            WHERE video_age_hours >= 6 AND video_age_hours < 114
            GROUP BY 1, 2, 3, 4)
        SELECT b.*, LN(1 + f.view_5d) - LN(1 + b.v_b) AS log_ratio
        FROM b JOIN video_5d_views f USING (video_id)
        WHERE f.view_5d IS NOT NULL AND f.is_shorts = 0
    ) TO '{RATIO}' (FORMAT parquet)""")
con.sql(f"CREATE OR REPLACE VIEW ratio AS SELECT * FROM '{RATIO}'")


def nowcast_sql(Dref):
    """[Dref−132h, Dref) 롱폼의 view_5d 추정. 비율은 Dref−132h 이전 영상으로만 학습 (미래 영상으로 배우지 않음)"""
    return f"""
        WITH r AS (
            SELECT age_b, MEDIAN(log_ratio) AS lr FROM ratio
            WHERE published_at < TIMESTAMP '{Dref}' - INTERVAL {LAG_H} HOUR GROUP BY 1),
        last AS (   -- 기준일 전 마지막 스냅샷 (나이는 수집 시각과 함께 증가하므로 MAX = 마지막)
            SELECT video_id, channel_id, published_at,
                   ARG_MAX(view_count, collected_at) AS v_now, MAX(video_age_hours) AS age_now
            FROM snap_early
            WHERE published_at >= TIMESTAMP '{Dref}' - INTERVAL {LAG_H} HOUR
              AND published_at <  TIMESTAMP '{Dref}' AND collected_at < TIMESTAMP '{Dref}'
            GROUP BY 1, 2, 3)
        SELECT l.video_id, l.channel_id, l.published_at, EXP(LN(1 + l.v_now) + r.lr) - 1 AS view_5d_hat, l.age_now
        FROM last l JOIN r ON r.age_b = LEAST(CAST(FLOOR(l.age_now / 6) * 6 AS INTEGER), 108)
        WHERE l.age_now >= 6                                                   -- 6h 미만은 추정하지 않음
    """


_V5 = {}
def v5_view(Dref, mode):
    """모드별 view_5d 테이블 이름을 돌려준다 (기준일·모드별로 한 번만 생성)"""
    key = (Dref, mode)
    if key in _V5:
        return _V5[key]
    name = f"v5_{mode}_{Dref.replace('-', '')}"
    cut = f"TIMESTAMP '{Dref}'" if mode == 'full' else f"TIMESTAMP '{Dref}' - INTERVAL {LAG_H} HOUR"
    sql = f"""SELECT video_id, channel_id, published_at, is_shorts, view_5d, like_5d, comment_5d, 0 AS is_hat
              FROM video_5d_views WHERE view_5d IS NOT NULL AND published_at < {cut}"""
    if mode == 'nowcast':
        sql += f""" UNION ALL
              SELECT video_id, channel_id, published_at, 0, view_5d_hat, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), 1
              FROM ({nowcast_sql(Dref)})"""
    con.sql(f"CREATE OR REPLACE TEMP TABLE {name} AS {sql}")
    _V5[key] = name
    return name

In [4]:
def make_features(Dref, mode='full'):
    """07과 같은 피처 23개 + 측정 상태 피처 3개 + past 이력 길이. view_5d는 v5_view(Dref, mode)에서만 읽는다"""
    v = v5_view(Dref, mode)
    f = con.sql(f"""
    WITH lv AS (   -- 롱폼 view_5d (모드에 따라 실제 / 실제+추정)
        SELECT channel_id, view_5d, like_5d, comment_5d, is_hat, published_at,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago
        FROM {v} WHERE is_shorts = 0),
    v5 AS (
        SELECT channel_id, COUNT(*) AS n_long_5d, MEDIAN(view_5d) AS past_med,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 7)   AS med_7,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 14)  AS med_14,
               MEDIAN(view_5d) FILTER (WHERE days_ago >= 14) AS med_old,
               MEDIAN(like_5d / NULLIF(view_5d, 0))    FILTER (WHERE is_hat = 0) AS like_rate,      -- 실제값만
               MEDIAN(comment_5d / NULLIF(view_5d, 0)) FILTER (WHERE is_hat = 0) AS comment_rate,
               STDDEV(LN(1 + view_5d))                       AS log_view_std,
               QUANTILE_CONT(LN(1 + view_5d), 0.1)           AS log_q10,
               QUANTILE_CONT(LN(1 + view_5d), 0.9)           AS log_q90,
               REGR_SLOPE(LN(1 + view_5d), -days_ago)        AS trend_slope,
               MIN(days_ago)                                 AS days_since_meas,
               AVG(is_hat) FILTER (WHERE days_ago < 14)      AS hat_share,       -- 최근 14일 중 추정치 비율
               DATE_DIFF('hour', MIN(published_at), TIMESTAMP '{Dref}') / 24.0 AS past_span_days   -- past 이력 길이
        FROM lv GROUP BY 1),
    sv AS (SELECT channel_id, MEDIAN(view_5d) AS short_med FROM {v} WHERE is_shorts = 1 GROUP BY 1),
    up AS (        -- 업로드 수: 업로드 즉시 알 수 있으므로 published_at < Dref 그대로
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS n_long_7,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 28 DAY) AS n_long_28,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               AVG(duration_sec) FILTER (WHERE is_shorts = 0) AS avg_dur,
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0 AS days_since_long
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1),
    games AS (
        SELECT v.channel_id, COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}' GROUP BY 1),
    unm AS (       -- 직전 132h 롱폼 중 view_5d(실제·추정)가 없는 영상 수
        SELECT x.channel_id, COUNT(*) AS n_unmeasured
        FROM videos x LEFT JOIN {v} y USING (video_id)
        WHERE x.is_shorts = 0 AND y.video_id IS NULL
          AND x.published_at >= TIMESTAMP '{Dref}' - INTERVAL {LAG_H} HOUR AND x.published_at < TIMESTAMP '{Dref}'
        GROUP BY 1),
    es AS (        -- 초기 조회수 기울기: 최근 14일 영상의 24h → 48h log 조회수 증가 (collected_at < Dref)
        SELECT channel_id, MEDIAN(LN(1 + v48) - LN(1 + v24)) AS early_slope
        FROM (SELECT video_id, channel_id,
                     ARG_MAX(view_count, collected_at) FILTER (WHERE video_age_hours <= 24) AS v24,
                     MAX(video_age_hours)              FILTER (WHERE video_age_hours <= 24) AS a24,
                     ARG_MAX(view_count, collected_at) FILTER (WHERE video_age_hours <= 48) AS v48,
                     MAX(video_age_hours)              FILTER (WHERE video_age_hours <= 48) AS a48
              FROM snap_early
              WHERE collected_at < TIMESTAMP '{Dref}' AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY
              GROUP BY 1, 2)
        WHERE a24 >= 18 AND a48 >= 42 GROUP BY 1)
    SELECT * FROM up LEFT JOIN v5 USING (channel_id) LEFT JOIN sv USING (channel_id)
                     LEFT JOIN games USING (channel_id) LEFT JOIN unm USING (channel_id) LEFT JOIN es USING (channel_id)
    """).df()
    f['n_unmeasured'] = f.n_unmeasured.fillna(0)
    f['n_long_5d']    = f.n_long_5d.fillna(0)
    lp = np.log1p(f.past_med)
    f['log_past_med']      = lp
    f['mom_7']             = np.log1p(f.med_7) - lp
    f['mom_14']            = np.log1p(f.med_14) - lp
    f['mom_old']           = np.log1p(f.med_14) - np.log1p(f.med_old)
    f['short_ratio']       = f.n_short_14 / (f.n_short_14 + f.n_long_14).replace(0, np.nan)
    f['log_short_med_rel'] = np.log1p(f.short_med) - lp
    f['has_game']          = (f.n_games > 0).astype(int)
    f['log_n_long_5d']     = np.log1p(f.n_long_5d)
    return f


def add_prev_features(df, Dref, prev_labels, mode='full'):
    """전주 라벨 + 전주 정답 구간 중 이미 아는 부분 [D−7, D)의 성과 (07과 같은 피처, v5_view 사용)"""
    v = v5_view(Dref, mode)
    P = (pd.Timestamp(Dref) - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
    pl = prev_labels[['channel_id', 'target']].rename(columns={'target': 'prev'})
    pp = con.sql(f"""SELECT channel_id, MEDIAN(view_5d) AS prev_past FROM {v}
                     WHERE is_shorts = 0 AND published_at < TIMESTAMP '{P}' GROUP BY 1""").df()
    kn = con.sql(f"""SELECT channel_id, COUNT(*) AS n_known, MEDIAN(view_5d) AS known_med FROM {v}
                     WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{P}' GROUP BY 1""").df()
    df = df.merge(pl, on='channel_id', how='left').merge(pp, on='channel_id', how='left').merge(kn, on='channel_id', how='left')
    df['n_known']        = df.n_known.fillna(0)
    df['has_prev']       = df.prev.notna().astype(int)
    df['prev_f']         = df.prev.fillna(0.2)
    df['log_prev_past']  = np.log1p(df.prev_past)
    df['known_rel_prev'] = np.log1p(df.known_med) - df.log_prev_past
    df['known_rel_now']  = np.log1p(df.known_med) - df.log_past_med
    df['prev_x_known']   = df.prev_f * (1 - df.known_rel_prev.fillna(0).clip(-3, 3) / 3)
    df['prev_x_lowpast'] = df.prev_f * (-df.log_past_med)
    return df


def make_labels(Dref, fut, fill_hat=False):
    """공식 라벨 규칙 재현. fill_hat=True면 view_5d가 없는 미래 롱폼을 09-01 nowcast 추정치로 채운다 (단계 D-1a)"""
    src = "SELECT channel_id, published_at, view_5d FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL"
    if fill_hat:
        src += f""" UNION ALL SELECT h.channel_id, h.published_at, h.view_5d_hat FROM ({nowcast_sql(D)}) h
                    WHERE h.video_id NOT IN (SELECT video_id FROM video_5d_views WHERE view_5d IS NOT NULL)"""
    g = con.sql(f"""
        WITH agg AS (
          SELECT channel_id,
                 MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS past_score,
                 COUNT(view_5d)  FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS n_past,
                 COALESCE(MEDIAN(view_5d) FILTER (WHERE published_at >= TIMESTAMP '{Dref}'
                       AND published_at < TIMESTAMP '{Dref}' + INTERVAL {fut} DAY), 0) AS future_score
          FROM ({src}) GROUP BY 1),
        scored AS (SELECT channel_id, LN(1 + future_score) - LN(1 + past_score) AS growth
                   FROM agg WHERE n_past >= 3 AND past_score >= 100)
        SELECT channel_id, CASE WHEN growth >= QUANTILE_CONT(growth, 0.8) OVER () THEN 1 ELSE 0 END AS target
        FROM scored""").df()
    g['ref_date'] = Dref
    return g


F23 = ['log_past_med', 'n_long_5d', 'like_rate', 'n_long_14', 'n_short_14', 'short_ratio', 'days_since_long',
       'mom_14', 'log_view_std', 'n_games', 'has_game', 'mom_7', 'mom_old', 'comment_rate',
       'log_q10', 'log_q90', 'trend_slope', 'days_since_meas', 'n_long_7', 'n_long_28', 'avg_dur',
       'log_short_med_rel', 'log_n_long_5d']
PREVF = ['prev_f', 'has_prev', 'log_prev_past', 'n_known', 'known_rel_prev', 'known_rel_now', 'prev_x_known', 'prev_x_lowpast']
PAIR  = F23 + PREVF                                  # 07 피처 31개
NOWF  = ['hat_share', 'n_unmeasured', 'early_slope'] # 단계 B 신규

In [5]:
_TRAIN = {}
def build_train(mode='full', fill_hat=False):
    """학습 기준일(07-26~08-05)별: D 시점 26일 라벨 + D 피처 + (D−7) 18일 재현 라벨 기반 역추론 피처"""
    key = (mode, fill_hat)
    if key not in _TRAIN:
        parts = []
        for d in TRAIN_DATES:
            P = (pd.Timestamp(d) - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
            df = make_labels(d, FUT, fill_hat)[['channel_id', 'target', 'ref_date']].merge(make_features(d, mode), on='channel_id', how='left')
            parts.append(add_prev_features(df, d, make_labels(P, FUT_PREV), mode))
        _TRAIN[key] = pd.concat(parts, ignore_index=True)
    return _TRAIN[key]


# 공식 08-25 라벨 · 채점 대상
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
PREV_OFF = tl[tl.row_id.str.endswith(PREV)][['channel_id', 'target']]
sample = pd.read_csv(f'{RAW}/sample_submission.csv', encoding='utf-8-sig')
sample['channel_id'] = sample.row_id.str.rsplit('_', n=1).str[0]

_TEST = {}
def build_test(mode='full'):
    """09-01 테스트 행: 학습과 같은 함수로 피처 생성 (테스트용 분기 없음)"""
    if mode not in _TEST:
        t = sample[['row_id', 'channel_id']].merge(make_features(D, mode), on='channel_id', how='left')
        _TEST[mode] = add_prev_features(t, D, PREV_OFF, mode)
    return _TEST[mode]

In [6]:
# ===== 모델 (07과 동일: HGB 5시드 평균 순위 + 로지스틱 순위의 평균) =====
def hgb(seed):
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, min_samples_leaf=30,
                                          l2_regularization=1.0, random_state=seed)

def logreg():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('m', LogisticRegression(max_iter=3000, C=0.1, class_weight='balanced'))])

def rank01(x):
    x = pd.Series(np.asarray(x, float))
    return rankdata(x.fillna(x.median())) / len(x)

def predict(tr, te, feats, w=None):
    """순위 점수(0~1)와 HGB 확률을 돌려준다. w = 학습 행 가중치(단계 D)"""
    p_hgb = np.mean([hgb(s).fit(tr[feats], tr.target, sample_weight=w).predict_proba(te[feats])[:, 1]
                     for s in range(5)], axis=0)
    p_lr = logreg().fit(tr[feats], tr.target, m__sample_weight=w).predict_proba(te[feats])[:, 1]
    return (rank01(p_hgb) + rank01(p_lr)) / 2, p_hgb

def validate(data, feats, weight_fn=None, return_preds=False):
    """07과 같은 검증: 평가일마다 (평가일 − 4일) 이하 기준일로 학습 → PR-AUC"""
    out, preds = {}, {}
    for td in EVAL_DATES:
        te = data[data.ref_date == td]
        tr = data[data.ref_date <= (pd.Timestamp(td) - pd.Timedelta(days=4)).strftime('%Y-%m-%d')]
        w = None if weight_fn is None else weight_fn(tr).values
        s, _ = predict(tr, te, feats, w)
        out[td[5:]] = average_precision_score(te.target, s)
        preds[td] = s
    out['평균'] = np.mean(list(out.values()))
    return (out, preds) if return_preds else out

VAL = []   # 단계별 검증 결과 누적
def log_val(name, res):
    VAL.append({'구성': name, **{k: round(v, 3) for k, v in res.items()}})
    display(pd.DataFrame(VAL).set_index('구성'))

In [7]:
# ===== 제출 파일 생성 (07 제출 셀과 같은 검사) =====
SUB07 = pd.read_csv(f'{SUB}/submit_07_prevlabel_inversion_hgb.csv')
SUBS = []   # 생성한 제출 파일 기록

def finalize(score, tie):
    """동점 깨기 → 0~1 정규화 → sample 순서로 정렬 → 제출 가이드 §3 검사"""
    s = np.asarray(score, float) + 1e-6 * rank01(tie)
    s = (s - s.min()) / (s.max() - s.min())
    sub = sample[['row_id']].merge(pd.DataFrame({'row_id': build_test('full').row_id, 'prediction': s}), on='row_id', how='left')
    assert list(sub.columns) == ['row_id', 'prediction'] and len(sub) == len(sample) == 694
    assert set(sub.row_id) == set(sample.row_id) and sub.row_id.is_unique
    assert sub.prediction.notna().all() and np.isfinite(sub.prediction).all() and sub.prediction.between(0, 1).all()
    return sub

def save_sub(sub, fname, desc, val_mean):
    path = f'{SUB}/{fname}'
    sub.to_csv(path, index=False, encoding='utf-8')
    rho = spearmanr(sub.merge(SUB07, on='row_id').prediction_x, sub.merge(SUB07, on='row_id').prediction_y)[0]
    SUBS.append({'파일': fname, '구성': desc, '검증 평균': round(val_mean, 3), '07과 Spearman': round(rho, 3)})
    print('저장:', fname, '| 07과 순위상관', round(rho, 3))

def fit_predict_test(mode, feats, weight_fn=None):
    """전체 학습 기준일로 학습 → 09-01 예측 (순위 점수, HGB 확률)"""
    tr, te = build_train(mode), build_test(mode)
    w = None if weight_fn is None else weight_fn(tr).values
    return predict(tr, te, feats, w)

## 2. 단계 A — 132h 마스킹(`trunc`) vs 07 방식(`full`)

지침대로 view_5d를 읽는 모든 곳(`lv`·`v5`, `sv`, `kn`, `pp`)을 `published_at < Dref − 132h`로 제한한 `trunc` 모드를 만들고,
07 방식(`full`)과 같은 검증 루프로 비교한다.

In [8]:
t0 = time.time()
log_val('07 원본 (full, 피처 31)', validate(build_train('full'), PAIR))
log_val('A: trunc (132h 마스킹, 피처 31)', validate(build_train('trunc'), PAIR))
print(f'{time.time() - t0:.0f}초')

,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.58,0.58,0.559


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488


43초


### 2.1. 분포 점검 — 학습(08-05) 행 vs 테스트(09-01) 행

지침은 "07(full)은 학습과 테스트 분포가 크게 다르다"고 가정했다. 두 모드에서 실제로 비교한다.

In [9]:
def dist_row(df, label):
    return {'구분': label,
            'days_since_meas 중앙값': round(df.days_since_meas.median(), 2),
            'n_known 평균': round(df.n_known.mean(), 2),
            'n_known>0 비율': round((df.n_known > 0).mean(), 3),
            'known_med 결측률': round(df.known_med.isna().mean(), 3),
            'med_7 결측률': round(df.med_7.isna().mean(), 3)}

dist = []
for mode in ['full', 'trunc']:
    tr05 = build_train(mode)
    dist.append(dist_row(tr05[tr05.ref_date == '2026-08-05'], f'{mode} · 학습 08-05'))
    dist.append(dist_row(build_test(mode), f'{mode} · 테스트 09-01'))
DIST = pd.DataFrame(dist).set_index('구분')
DIST

,days_since_meas 중앙값,n_known 평균,n_known>0 비율,known_med 결측률,med_7 결측률
구분,,,,,
full · 학습 08-05,1.21,4.97,0.946,0.054,0.054
full · 테스트 09-01,1.71,4.45,0.862,0.138,0.138
trunc · 학습 08-05,6.46,1.17,0.676,0.324,0.328
trunc · 테스트 09-01,6.96,0.94,0.507,0.493,0.497


**해석**: `full`에서도 학습 08-05와 테스트 09-01의 측정 상태가 비슷하다(전제와 달리 테스트에도 최근 영상의 view_5d가 있음).
`trunc`는 두 쪽 모두 최근 5.5일 정보를 버리므로 분포는 같지만, 정보량이 줄어 검증 점수가 떨어진다.

In [10]:
s, ph = fit_predict_test('trunc', PAIR)
save_sub(finalize(s, ph), 'submit_08A_trunc132.csv', 'A: 132h 마스킹, 피처 31', VAL[-1]['평균'])

저장: submit_08A_trunc132.csv | 07과 순위상관 0.848


## 3. 단계 B — 최근 5.5일 view_5d nowcast

### 3.1. nowcast 정확도

08-10 이전 영상으로 나이별 비율을 배우고, 08-10 ~ 08-25 영상(실제 view_5d 있음)에 "추정했다고 치고" 비교한다.

In [11]:
acc = con.sql("""
    WITH r AS (SELECT age_b, MEDIAN(log_ratio) AS lr FROM ratio
               WHERE published_at < TIMESTAMP '2026-08-10' GROUP BY 1),
    t AS (SELECT x.age_b, x.log_ratio, r.lr FROM ratio x JOIN r USING (age_b)
          WHERE x.published_at >= TIMESTAMP '2026-08-10' AND x.published_at < TIMESTAMP '2026-08-25')
    SELECT age_b AS "나이(h)", COUNT(*) AS n,
           ROUND(AVG(ABS(log_ratio - lr)), 3) AS "log MAE",
           ROUND(MEDIAN(lr), 3) AS "보정비율(log)"
    FROM t GROUP BY 1 ORDER BY 1""").df()
# 상관계수: 추정 log view_5d vs 실제 log view_5d
corr = con.sql("""
    WITH r AS (SELECT age_b, MEDIAN(log_ratio) AS lr FROM ratio WHERE published_at < TIMESTAMP '2026-08-10' GROUP BY 1)
    SELECT x.age_b, CORR(LN(1 + x.v_b) + r.lr, LN(1 + x.v_b) + x.log_ratio) AS corr
    FROM ratio x JOIN r USING (age_b)
    WHERE x.published_at >= TIMESTAMP '2026-08-10' AND x.published_at < TIMESTAMP '2026-08-25'
    GROUP BY 1""").df().rename(columns={'age_b': '나이(h)'})
ACC = acc.merge(corr, on='나이(h)').round(3)
display(ACC[ACC['나이(h)'].isin([6, 12, 24, 36, 48, 72, 96, 108])])
print('24h 이상 log MAE 최대:', ACC[ACC['나이(h)'] >= 24]['log MAE'].max())

,나이(h),n,log MAE,보정비율(log),corr
0,6,2472,0.674,0.641,0.845
1,12,5132,0.469,0.487,0.921
3,24,7039,0.266,0.279,0.969
5,36,7046,0.208,0.191,0.976
7,48,7048,0.149,0.124,0.985
11,72,7049,0.083,0.054,0.993
15,96,7051,0.032,0.018,0.998
17,108,7052,0.012,0.005,0.999


24h 이상 log MAE 최대: 0.266


### 3.2. 검증 — nowcast 모드

`nowcast` 모드(실제 `< Dref−132h` + 추정 `[Dref−132h, Dref)`)에 측정 상태 피처(`hat_share`, `n_unmeasured`, `early_slope`)를 더한다.
비교를 위해 `full` 모드에 같은 신규 피처를 넣은 경우도 본다(`early_slope`는 `collected_at < Dref` 스냅샷만 쓰므로 `full`에서도 유효).

In [12]:
log_val('B: nowcast (피처 31)', validate(build_train('nowcast'), PAIR))
log_val('B: nowcast + 신규 3 (피처 34)', validate(build_train('nowcast'), PAIR + NOWF))
log_val('full + 신규 3 (피처 34)', validate(build_train('full'), PAIR + NOWF))

,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560


In [13]:
# 테스트(09-01)에서 아는 부분 [08-25, 09-01) 커버리지 — 07(full)은 598채널
cov = []
for mode in ['full', 'trunc', 'nowcast']:
    t = build_test(mode)
    v = v5_view(D, mode)
    span = con.sql(f"SELECT MIN(published_at), MAX(published_at) FROM {v} WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{PREV}'").fetchone()
    cov.append({'모드': mode, 'n_known>0 채널': int((t.n_known > 0).sum()), 'known 구간 시작': span[0], 'known 구간 끝': span[1]})
display(pd.DataFrame(cov))

,모드,n_known>0 채널,known 구간 시작,known 구간 끝
0,full,598,2026-08-25 00:00:20,2026-08-31 23:59:27
1,trunc,352,2026-08-25 00:00:20,2026-08-26 11:58:18
2,nowcast,591,2026-08-25 00:00:20,2026-08-31 03:03:58


In [14]:
s, ph = fit_predict_test('nowcast', PAIR + NOWF)
save_sub(finalize(s, ph), 'submit_08B_nowcast.csv', 'B: nowcast + 신규 3', VAL[-2]['평균'])

저장: submit_08B_nowcast.csv | 07과 순위상관 0.919


### 3.3. A·B 결론 → 이후 단계는 `full` 모드로 진행

- nowcast 자체는 정확하다(24h 이후 log MAE ≤ 0.27, 상관 ≥ 0.97).
- 그래도 **실제 view_5d를 쓰는 `full`이 가장 높다.** 테스트에도 실제 값이 있으므로 추정할 이유가 없다.
- 마스킹(A)은 정보만 버려 검증이 크게 떨어진다. → **채택: `full` 모드.** 이후 C·D·E는 `full` 위에 쌓는다.

In [15]:
MODE = 'full'
v07  = [r for r in VAL if r['구성'].startswith('07')][0]
vnew = [r for r in VAL if r['구성'].startswith('full +')][0]
USE_NOW = vnew['평균'] > v07['평균'] + 0.003           # 잡음(±0.003) 이상으로 오를 때만 채택
best  = vnew if USE_NOW else v07
FEATS = PAIR + NOWF if USE_NOW else PAIR
print('채택 모드:', MODE, '| 피처 수:', len(FEATS), '| 기준 검증:', best['평균'])

채택 모드: full | 피처 수: 31 | 기준 검증: 0.559


## 4. 단계 C — 채널 스냅샷 피처

`channel_snapshots`(하루 2회)에서 `collected_at < Dref`만 사용. 각 시점 값은 "그 시점 이전 마지막 스냅샷"(ASOF).

| 피처 | 의미 |
|---|---|
| `sub_growth_7`, `sub_growth_14` | ln(구독자) 변화량 |
| `view_delta_3`, `view_delta_7` | ln(채널 누적 조회수) 변화량 |
| `view_rel_3`, `view_rel_7` | 일평균 조회수 증가분(log) − `log_past_med` |
| `view_accel` | 최근 3일 일평균 증가 vs 그 이전 11일 일평균 증가 (log 차) |
| `vcount_delta_14` | 영상 수 변화량 |
| `ch_hist_days`, `ch_missing_14` | 수집 이력 길이, 14일 전 스냅샷 없음 표시 |

In [16]:
def ch_feats(Dref):
    c = con.sql(f"""
        SELECT channel_id,
               ARG_MAX(subscriber_count, collected_at) AS s0,
               ARG_MAX(subscriber_count, collected_at) FILTER (WHERE collected_at < TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS s7,
               ARG_MAX(subscriber_count, collected_at) FILTER (WHERE collected_at < TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS s14,
               ARG_MAX(total_view_count, collected_at) AS t0,
               ARG_MAX(total_view_count, collected_at) FILTER (WHERE collected_at < TIMESTAMP '{Dref}' - INTERVAL 3 DAY)  AS t3,
               ARG_MAX(total_view_count, collected_at) FILTER (WHERE collected_at < TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS t7,
               ARG_MAX(total_view_count, collected_at) FILTER (WHERE collected_at < TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS t14,
               ARG_MAX(video_count, collected_at) AS c0,
               ARG_MAX(video_count, collected_at) FILTER (WHERE collected_at < TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS c14,
               DATE_DIFF('hour', MIN(collected_at), TIMESTAMP '{Dref}') / 24.0 AS ch_hist_days
        FROM channel_snapshots WHERE collected_at < TIMESTAMP '{Dref}' GROUP BY 1""").df()
    l = lambda x: np.log1p(x.astype(float).clip(lower=0))
    c['sub_growth_7']    = l(c.s0) - l(c.s7)
    c['sub_growth_14']   = l(c.s0) - l(c.s14)
    c['view_delta_3']    = l(c.t0) - l(c.t3)
    c['view_delta_7']    = l(c.t0) - l(c.t7)
    c['_d3']             = l((c.t0 - c.t3) / 3)          # 최근 3일 일평균 증가 (log)
    c['_d7']             = l((c.t0 - c.t7) / 7)
    c['view_accel']      = c._d3 - l((c.t3 - c.t14) / 11)
    c['vcount_delta_14'] = c.c0 - c.c14
    c['ch_missing_14']   = c.s14.isna().astype(int)
    return c[['channel_id', 'sub_growth_7', 'sub_growth_14', 'view_delta_3', 'view_delta_7', '_d3', '_d7',
              'view_accel', 'vcount_delta_14', 'ch_hist_days', 'ch_missing_14']]

def add_ch(df, Dref):
    df = df.merge(ch_feats(Dref), on='channel_id', how='left')
    df['view_rel_3'] = df._d3 - df.log_past_med
    df['view_rel_7'] = df._d7 - df.log_past_med
    return df

CHF = ['sub_growth_7', 'sub_growth_14', 'view_delta_3', 'view_delta_7', 'view_rel_3', 'view_rel_7',
       'view_accel', 'vcount_delta_14', 'ch_hist_days', 'ch_missing_14']

train_c = pd.concat([add_ch(g, d) for d, g in build_train(MODE).groupby('ref_date')], ignore_index=True)
test_c  = add_ch(build_test(MODE), D)
print('채널 피처 결측률 (학습 / 테스트):')
display(pd.DataFrame({'학습': train_c[CHF].isna().mean(), '테스트': test_c[CHF].isna().mean()}).round(3).T)
log_val('C: + 채널 스냅샷 10', validate(train_c, FEATS + CHF))

채널 피처 결측률 (학습 / 테스트):


,sub_growth_7,sub_growth_14,view_delta_3,view_delta_7,view_rel_3,view_rel_7,view_accel,vcount_delta_14,ch_hist_days,ch_missing_14
학습,0.552,0.552,0.552,0.552,0.552,0.552,0.552,0.552,0.552,0.552
테스트,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560
C: + 채널 스냅샷 10,0.545,0.543,0.539,0.587,0.589,0.561


In [17]:
# HGB permutation importance (학습 ≤ 08-01 → 평가 08-05, PR-AUC 기준)
tr_i, te_i = train_c[train_c.ref_date <= '2026-08-01'], train_c[train_c.ref_date == '2026-08-05']
m_i = hgb(SEED).fit(tr_i[FEATS + CHF], tr_i.target)
pi = permutation_importance(m_i, te_i[FEATS + CHF], te_i.target, scoring='average_precision', n_repeats=5, random_state=SEED)
IMP = pd.Series(pi.importances_mean, index=FEATS + CHF).sort_values(ascending=False).head(15).round(4)
IMP

log_past_med         0.0669
log_view_std         0.0440
like_rate            0.0399
n_long_28            0.0270
n_short_14           0.0262
log_short_med_rel    0.0252
view_rel_3           0.0223
known_rel_prev       0.0208
prev_x_known         0.0186
prev_f               0.0182
avg_dur              0.0178
comment_rate         0.0156
mom_7                0.0153
n_long_14            0.0100
n_games              0.0072
dtype: float64

In [18]:
USE_C = VAL[-1]['평균'] > best['평균'] + 0.003          # 잡음 이상으로 오를 때만 채택
FEATS_C = FEATS + CHF if USE_C else FEATS
TRAIN, TEST = (train_c, test_c)
print('채널 피처 채택:', USE_C)

def predict_full(feats, weight_fn=None, data=None):
    tr = TRAIN if data is None else data
    w = None if weight_fn is None else weight_fn(tr).values
    return predict(tr, TEST, feats, w)

s, ph = predict_full(FEATS + CHF)
save_sub(finalize(s, ph), 'submit_08C_full_channel.csv', f'full 피처 {len(FEATS)} + 채널 10', [r for r in VAL if r['구성'].startswith('C:')][0]['평균'])

채널 피처 채택: False


저장: submit_08C_full_channel.csv | 07과 순위상관 0.938


## 5. 단계 D — 학습 라벨 보정 · 가중치

- **D-1(a)** 미래 영상 중 view_5d가 없는 것을 09-01 nowcast로 채운 라벨
- **D-1(b)** 미래 구간이 08-26 12시를 넘는 기준일(08-01 이후) 가중치 0.5
- **D-2** `past_span_days`(past 이력 길이) 피처 추가 / 늦은 기준일일수록 가중치↑(선형 1→2)

참고: §0에서 본 것처럼 08-26 이후 영상도 view_5d가 약 96% 있어(다른 날짜와 같은 수준), D-1이 고칠 결측은 크지 않다.

In [19]:
fill = build_train(MODE, fill_hat=True)[['channel_id', 'ref_date', 'target']].rename(columns={'target': 'target_fill'})
train_fill = TRAIN.merge(fill, on=['channel_id', 'ref_date'], how='left')
print('D-1(a) 라벨 변경 행:', int((train_fill.target_fill.notna() & (train_fill.target_fill != train_fill.target)).sum()), '/', len(train_fill))
train_fill = train_fill[train_fill.target_fill.notna()].assign(target=lambda d: d.target_fill.astype(int))

late_half = lambda tr: pd.Series(np.where(tr.ref_date >= '2026-08-01', 0.5, 1.0), index=tr.index)
DATE_IDX  = {d: i for i, d in enumerate(TRAIN_DATES)}
linear    = lambda tr: 1 + tr.ref_date.map(DATE_IDX) / (len(TRAIN_DATES) - 1)

D_OPTS = {
    'D-1(a) 라벨 nowcast 채움': dict(data=train_fill, feats=FEATS_C, w=None),
    'D-1(b) 08-01 이후 가중치 0.5': dict(data=TRAIN, feats=FEATS_C, w=late_half),
    'D-2 past_span_days 추가': dict(data=TRAIN, feats=FEATS_C + ['past_span_days'], w=None),
    'D-2 선형 가중치(늦을수록↑)': dict(data=TRAIN, feats=FEATS_C, w=linear),
}
base_c = VAL[-1]['평균'] if USE_C else best['평균']
d_res = {}
for name, o in D_OPTS.items():
    r = validate(o['data'], o['feats'], o['w'])
    d_res[name] = r['평균']
    log_val(name, r)

D-1(a) 라벨 변경 행: 8 / 5102


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560
C: + 채널 스냅샷 10,0.545,0.543,0.539,0.587,0.589,0.561
D-1(a) 라벨 nowcast 채움,0.546,0.552,0.536,0.581,0.582,0.560


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560
C: + 채널 스냅샷 10,0.545,0.543,0.539,0.587,0.589,0.561
D-1(a) 라벨 nowcast 채움,0.546,0.552,0.536,0.581,0.582,0.560
D-1(b) 08-01 이후 가중치 0.5,0.546,0.552,0.538,0.580,0.572,0.558


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560
C: + 채널 스냅샷 10,0.545,0.543,0.539,0.587,0.589,0.561
D-1(a) 라벨 nowcast 채움,0.546,0.552,0.536,0.581,0.582,0.560
D-1(b) 08-01 이후 가중치 0.5,0.546,0.552,0.538,0.580,0.572,0.558
D-2 past_span_days 추가,0.548,0.549,0.538,0.582,0.573,0.558


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560
C: + 채널 스냅샷 10,0.545,0.543,0.539,0.587,0.589,0.561
D-1(a) 라벨 nowcast 채움,0.546,0.552,0.536,0.581,0.582,0.560
D-1(b) 08-01 이후 가중치 0.5,0.546,0.552,0.538,0.580,0.572,0.558
D-2 past_span_days 추가,0.548,0.549,0.538,0.582,0.573,0.558


In [20]:
best_d = max(d_res, key=d_res.get)
USE_D = d_res[best_d] > base_c + 0.003
D_CHOICE = D_OPTS[best_d] if USE_D else dict(data=TRAIN, feats=FEATS_C, w=None)
print('D 채택:', best_d if USE_D else '없음 (모든 옵션이 잡음 수준 이하)', '| 기준', round(base_c, 3))

# D 채택안으로 테스트 예측 (라벨 채움은 학습 데이터만 바뀜)
FINAL_FEATS, FINAL_W, FINAL_DATA = D_CHOICE['feats'], D_CHOICE['w'], D_CHOICE['data']
s_model, ph_model = predict_full(FINAL_FEATS, FINAL_W, FINAL_DATA)
save_sub(finalize(s_model, ph_model), 'submit_08D_model.csv',
         f"C{'+' if USE_C else '−'} / D: {best_d if USE_D else '없음'}", max(d_res[best_d], base_c) if USE_D else base_c)

D 채택: 없음 (모든 옵션이 잡음 수준 이하) | 기준 0.559


저장: submit_08D_model.csv | 07과 순위상관 1.0


## 6. 단계 E — 라벨 공식 몬테카를로 시뮬레이션 (+ 전주 라벨 조건부)

채널마다 (기준일 시점 정보만으로) `mu`(최근 log view_5d, 최근성 가중 평균을 전체 평균으로 축소), `sigma`(축소한 표준편차),
`lam`(최근 28일 롱폼 업로드율)을 정하고 S=2000회 시뮬레이션한다.

- `known` = [PREV, D) 롱폼의 실제 log view_5d (고정)
- `fut1` ~ Poisson(lam × 11일)개 → [D, PREV+18일) : 전주 정답 구간과 겹치는 부분 (08-25 + 18일 = 09-12 → 11일)
- `fut2` ~ Poisson(lam × 15일)개 → 나머지
- `g_prev = median(known ∪ fut1) − log past_prev`, `g_now = median(fut1 ∪ fut2) − log past_now` (영상 없으면 future = 0)
- 시뮬마다 채널 간 80% 분위수로 임계값 → `y = g_now ≥ thr_now`
- 전주 라벨과 시뮬 결과가 맞으면 가중치 1, 틀리면 `eps` → `P = Σ w·y / Σ w`

튜닝 후보는 소수로 제한: 축소 강도 k ∈ {2, 5, 15}, eps ∈ {0.02, 0.05, 0.2}.

In [21]:
FUT1 = 11                       # 전주 정답 구간과 겹치는 일수 (7일 간격, 18일 창)
FUT2 = FUT - FUT1
KMAX = 40                       # 구간당 영상 수 상한 (중앙값 계산용, 그 이상은 결과 거의 동일)

def mc_inputs(Dref, rows, k_shrink, k_sig=5):
    """채널별 시뮬레이션 입력 (Dref 시점 정보만)"""
    v = v5_view(Dref, MODE)
    P = (pd.Timestamp(Dref) - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
    lv = con.sql(f"""SELECT channel_id, LN(1 + view_5d) AS lv, published_at,
                            DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago
                     FROM {v} WHERE is_shorts = 0""").df()
    rec = lv[lv.days_ago < 28].copy()
    rec['w'] = np.exp(-rec.days_ago / 14)                  # 최근일수록 가중
    agg = rec.groupby('channel_id').apply(lambda g: pd.Series({'m': np.average(g.lv, weights=g.w), 'n_eff': g.w.sum()}))
    sd = lv.groupby('channel_id').lv.agg(['std', 'count'])
    m_glob, s_glob = rec.lv.mean(), sd['std'].median()
    x = rows[['channel_id', 'log_past_med', 'log_prev_past', 'n_long_28', 'prev']].merge(agg, on='channel_id', how='left').merge(sd, on='channel_id', how='left')
    x['n_eff'] = x.n_eff.fillna(0)
    x['mu'] = (x.n_eff * x.m.fillna(m_glob) + k_shrink * m_glob) / (x.n_eff + k_shrink)
    x['sigma'] = ((x['count'].fillna(0) * x['std'].fillna(s_glob) + k_sig * s_glob) / (x['count'].fillna(0) + k_sig)).clip(lower=0.2)
    hist = con.sql(f"SELECT channel_id, DATE_DIFF('hour', MIN(published_at), TIMESTAMP '{Dref}') / 24.0 AS h FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1").df()
    x = x.merge(hist, on='channel_id', how='left')
    x['lam'] = (x.n_long_28.fillna(0) / x.h.fillna(28).clip(1, 28))
    kn = lv[lv.published_at >= pd.Timestamp(P)].groupby('channel_id').lv.apply(np.array)
    x['known'] = x.channel_id.map(kn)
    return x


def mc_sim(x, S=2000, seed=SEED):
    """S×채널 시뮬레이션 → (G_prev, G_now) 행렬"""
    rng = np.random.default_rng(seed)
    N = len(x)
    G_prev, G_now = np.empty((S, N)), np.empty((S, N))
    for j, r in enumerate(x.itertuples(index=False)):
        k1 = np.minimum(rng.poisson(r.lam * FUT1, S), KMAX)
        k2 = np.minimum(rng.poisson(r.lam * FUT2, S), KMAX)
        A = rng.normal(r.mu, r.sigma, (S, max(k1.max(), 1)))
        B = rng.normal(r.mu, r.sigma, (S, max(k2.max(), 1)))
        known = r.known if isinstance(r.known, np.ndarray) else np.empty(0)
        lpp = r.log_prev_past if pd.notna(r.log_prev_past) else r.log_past_med
        for k in np.unique(k1):                                     # g_prev: known ∪ fut1
            idx = np.where(k1 == k)[0]
            if len(known) + k == 0:
                G_prev[idx, j] = -lpp
            else:
                G_prev[idx, j] = np.median(np.hstack([np.broadcast_to(known, (len(idx), len(known))), A[idx, :k]]), axis=1) - lpp
        pair = k1 * (KMAX + 1) + k2
        for pk in np.unique(pair):                                  # g_now: fut1 ∪ fut2
            idx = np.where(pair == pk)[0]
            a, b = divmod(pk, KMAX + 1)
            if a + b == 0:
                G_now[idx, j] = -r.log_past_med
            else:
                G_now[idx, j] = np.median(np.hstack([A[idx, :a], B[idx, :b]]), axis=1) - r.log_past_med
    return G_prev, G_now


def mc_prob(x, G_prev, G_now, eps):
    y = G_now >= np.quantile(G_now, 0.8, axis=1, keepdims=True)
    has = x.prev.notna().values
    w = np.ones_like(G_now)
    if has.any():
        thr = np.quantile(G_prev[:, has], 0.8, axis=1, keepdims=True)
        match = (G_prev[:, has] >= thr) == (x.prev.values[has] == 1)
        w[:, has] = np.where(match, 1.0, eps)
    return (w * y).sum(0) / w.sum(0)

In [22]:
# 백테스트: 평가 기준일마다 MC 확률 (prev = 07과 같은 재현 라벨). 축소 강도별로 한 번씩 시뮬레이션
t0 = time.time()
MC = {}   # (k, eps) -> {평가일: 확률}
for k in [2, 5, 15]:
    for td in EVAL_DATES:
        rows = FINAL_DATA[FINAL_DATA.ref_date == td]
        x = mc_inputs(td, rows, k)
        Gp, Gn = mc_sim(x)
        for eps in [0.02, 0.05, 0.2]:
            MC.setdefault((k, eps), {})[td] = mc_prob(x, Gp, Gn, eps)
print(f'시뮬레이션 {time.time() - t0:.0f}초')

mc_tab = []
for (k, eps), pr in MC.items():
    r = {td[5:]: average_precision_score(FINAL_DATA[FINAL_DATA.ref_date == td].target, p) for td, p in pr.items()}
    mc_tab.append({'k': k, 'eps': eps, **{a: round(b, 3) for a, b in r.items()}, '평균': round(np.mean(list(r.values())), 3)})
MC_TAB = pd.DataFrame(mc_tab).sort_values('평균', ascending=False)
display(MC_TAB.head(5))
K_BEST, EPS_BEST = MC_TAB.iloc[0][['k', 'eps']]
K_BEST = int(K_BEST)
print('MC 채택 파라미터: k =', K_BEST, ', eps =', EPS_BEST)

시뮬레이션 52초


,k,eps,08-01,08-02,08-03,08-04,08-05,평균
0,2,0.02,0.343,0.360,0.351,0.348,0.343,0.349
1,2,0.05,0.341,0.357,0.348,0.345,0.340,0.346
2,2,0.20,0.336,0.351,0.342,0.339,0.331,0.340
3,5,0.02,0.318,0.329,0.319,0.328,0.318,0.322
4,5,0.05,0.317,0.327,0.318,0.327,0.316,0.321


MC 채택 파라미터: k = 2 , eps = 0.02


축소 강도는 후보 최소값(k=2)에서 가장 높았다(약한 축소일수록 유리). 다만 MC 단독 점수가 모델보다 크게 낮아,
후보를 넓혀도 아래 블렌딩 결론이 바뀔 가능성은 작다고 보고 지침대로 후보를 늘리지 않았다.

In [23]:
# 블렌딩: (A~D 최종 모델 순위)와 (MC 확률 순위)의 가중 평균
res_m, preds_m = validate(FINAL_DATA, FINAL_FEATS, FINAL_W, return_preds=True)
blend_rows = []
for a in [0, 0.25, 0.5, 0.75, 1.0]:
    r = {}
    for td in EVAL_DATES:
        y = FINAL_DATA[FINAL_DATA.ref_date == td].target
        s = (1 - a) * rank01(preds_m[td]) + a * rank01(MC[(K_BEST, EPS_BEST)][td])
        r[td[5:]] = average_precision_score(y, s)
    r['평균'] = np.mean(list(r.values()))
    name = 'E: MC 단독' if a == 1 else ('E: 모델 단독 (A~D)' if a == 0 else f'E: 블렌드 MC {a:.2f}')
    blend_rows.append((a, r['평균']))
    log_val(name, r)
A_BEST = max([b for b in blend_rows if b[0] < 1], key=lambda b: b[1])[0]
print('채택 MC 가중치:', A_BEST)

,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560
C: + 채널 스냅샷 10,0.545,0.543,0.539,0.587,0.589,0.561
D-1(a) 라벨 nowcast 채움,0.546,0.552,0.536,0.581,0.582,0.560
D-1(b) 08-01 이후 가중치 0.5,0.546,0.552,0.538,0.580,0.572,0.558
D-2 past_span_days 추가,0.548,0.549,0.538,0.582,0.573,0.558


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560
C: + 채널 스냅샷 10,0.545,0.543,0.539,0.587,0.589,0.561
D-1(a) 라벨 nowcast 채움,0.546,0.552,0.536,0.581,0.582,0.560
D-1(b) 08-01 이후 가중치 0.5,0.546,0.552,0.538,0.580,0.572,0.558
D-2 past_span_days 추가,0.548,0.549,0.538,0.582,0.573,0.558


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560
C: + 채널 스냅샷 10,0.545,0.543,0.539,0.587,0.589,0.561
D-1(a) 라벨 nowcast 채움,0.546,0.552,0.536,0.581,0.582,0.560
D-1(b) 08-01 이후 가중치 0.5,0.546,0.552,0.538,0.580,0.572,0.558
D-2 past_span_days 추가,0.548,0.549,0.538,0.582,0.573,0.558


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560
C: + 채널 스냅샷 10,0.545,0.543,0.539,0.587,0.589,0.561
D-1(a) 라벨 nowcast 채움,0.546,0.552,0.536,0.581,0.582,0.560
D-1(b) 08-01 이후 가중치 0.5,0.546,0.552,0.538,0.580,0.572,0.558
D-2 past_span_days 추가,0.548,0.549,0.538,0.582,0.573,0.558


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560
C: + 채널 스냅샷 10,0.545,0.543,0.539,0.587,0.589,0.561
D-1(a) 라벨 nowcast 채움,0.546,0.552,0.536,0.581,0.582,0.560
D-1(b) 08-01 이후 가중치 0.5,0.546,0.552,0.538,0.580,0.572,0.558
D-2 past_span_days 추가,0.548,0.549,0.538,0.582,0.573,0.558


채택 MC 가중치: 0


## 7. 최종 제출

In [24]:
x_test = mc_inputs(D, TEST, K_BEST)
Gp, Gn = mc_sim(x_test)
p_mc = mc_prob(x_test, Gp, Gn, EPS_BEST)
final = (1 - A_BEST) * rank01(s_model) + A_BEST * rank01(p_mc)
tag = f"full{'_ch' if USE_C else ''}{'_D' if USE_D else ''}_mc{int(A_BEST * 100)}"
save_sub(finalize(final, ph_model), f'submit_08_{tag}.csv', f'최종: 모델 + MC {A_BEST}', max(b[1] for b in blend_rows if b[0] == A_BEST))
save_sub(finalize(p_mc, ph_model), 'submit_08E_mc_only.csv', f'MC 단독 (k={K_BEST}, eps={EPS_BEST})',
         [r for r in VAL if r['구성'] == 'E: MC 단독'][0]['평균'])

저장: submit_08_full_mc0.csv | 07과 순위상관 1.0
저장: submit_08E_mc_only.csv | 07과 순위상관 0.797


## 8. 정리

In [25]:
print('① 단계별 검증 PR-AUC (평가 기준일 08-01 ~ 08-05, 07과 같은 루프)')
display(pd.DataFrame(VAL).set_index('구성'))
print('\n② 학습-테스트 분포 점검 (08-05 학습 행 vs 09-01 테스트 행)')
display(DIST)
print('\n③ nowcast 정확도 (08-10 이전으로 학습 → 08-10~08-25 영상 평가)')
display(ACC[ACC['나이(h)'].isin([12, 24, 48, 72, 108])].set_index('나이(h)'))
print('\n④ 중요 피처 상위 15 (HGB permutation importance, 08-05)')
display(IMP.to_frame('PR-AUC 감소'))
print('\n⑤ 생성한 제출 파일')
display(pd.DataFrame(SUBS))

① 단계별 검증 PR-AUC (평가 기준일 08-01 ~ 08-05, 07과 같은 루프)


,08-01,08-02,08-03,08-04,08-05,평균
구성,,,,,,
"07 원본 (full, 피처 31)",0.546,0.552,0.538,0.580,0.580,0.559
"A: trunc (132h 마스킹, 피처 31)",0.491,0.494,0.467,0.489,0.498,0.488
B: nowcast (피처 31),0.498,0.532,0.501,0.554,0.532,0.523
B: nowcast + 신규 3 (피처 34),0.489,0.526,0.496,0.545,0.529,0.517
full + 신규 3 (피처 34),0.547,0.562,0.533,0.576,0.580,0.560
C: + 채널 스냅샷 10,0.545,0.543,0.539,0.587,0.589,0.561
D-1(a) 라벨 nowcast 채움,0.546,0.552,0.536,0.581,0.582,0.560
D-1(b) 08-01 이후 가중치 0.5,0.546,0.552,0.538,0.580,0.572,0.558
D-2 past_span_days 추가,0.548,0.549,0.538,0.582,0.573,0.558



② 학습-테스트 분포 점검 (08-05 학습 행 vs 09-01 테스트 행)


,days_since_meas 중앙값,n_known 평균,n_known>0 비율,known_med 결측률,med_7 결측률
구분,,,,,
full · 학습 08-05,1.21,4.97,0.946,0.054,0.054
full · 테스트 09-01,1.71,4.45,0.862,0.138,0.138
trunc · 학습 08-05,6.46,1.17,0.676,0.324,0.328
trunc · 테스트 09-01,6.96,0.94,0.507,0.493,0.497



③ nowcast 정확도 (08-10 이전으로 학습 → 08-10~08-25 영상 평가)


,n,log MAE,보정비율(log),corr
나이(h),,,,
12,5132,0.469,0.487,0.921
24,7039,0.266,0.279,0.969
48,7048,0.149,0.124,0.985
72,7049,0.083,0.054,0.993
108,7052,0.012,0.005,0.999



④ 중요 피처 상위 15 (HGB permutation importance, 08-05)


,PR-AUC 감소
log_past_med,0.0669
log_view_std,0.0440
like_rate,0.0399
n_long_28,0.0270
n_short_14,0.0262
log_short_med_rel,0.0252
view_rel_3,0.0223
known_rel_prev,0.0208
prev_x_known,0.0186
prev_f,0.0182



⑤ 생성한 제출 파일


,파일,구성,검증 평균,07과 Spearman
0,submit_08A_trunc132.csv,"A: 132h 마스킹, 피처 31",0.488,0.848
1,submit_08B_nowcast.csv,B: nowcast + 신규 3,0.517,0.919
2,submit_08C_full_channel.csv,full 피처 31 + 채널 10,0.561,0.938
3,submit_08D_model.csv,C− / D: 없음,0.559,1.000
4,submit_08_full_mc0.csv,최종: 모델 + MC 0,0.559,1.000
5,submit_08E_mc_only.csv,"MC 단독 (k=2, eps=0.02)",0.349,0.797


**⑥ 남은 위험 요소**
- **08-25 공식 라벨 사용 승인 여부** — 09-01 이후 영상 성과로 계산된 값. 운영진 확인 전에는 07·08 계열 제출이 문제 될 수 있다.
- **리더보드 ±0.03 흔들림** — 694행 · 양성 약 139개. 검증 차이 0.01 이하는 잡음으로 본다.
- **지침 전제와 데이터 불일치** — 테스트 시점에도 최근 132h 영상의 view_5d가 약 96% 있어, 단계 A(마스킹)·B(nowcast)는 정보 손실로 검증이 떨어졌다. 이후 단계는 `full` 모드 기준.
- **검증 기준일 5개가 서로 겹침** — 08-01 ~ 08-05는 정답 구간이 대부분 겹쳐 독립 평가가 아니다. 26일 창을 깨끗하게 검증할 기준일이 데이터 범위상 없다.
- **MC 시뮬레이션 가정** — 미래 영상 조회수를 채널별 정규분포로 가정. 바이럴(꼬리) 영상은 과소평가된다.